# AIR: amortised importance sampling over an explicit transform latent

## Description

<!-- Written before the first run (2026-11-02); kept up to date. Spec: docs/a2b_registration_proposal_2026-11-02.md (the
user's registration proposal; its V0–V8 criteria are the pre-registered criteria). Background:
docs/a2b_bayes_tree_2026-10-29.md. Issue #19. -->

**Question.** Can the A↔B transfer be solved with an explicit transform latent, where every learned part targets an
exact Bayesian factor and the parts are combined by importance sampling? Earlier learned registration in this repo
(notebook 09) did not register. The proposal replaces "learn to move particles" with three parts:
- a **transported decoder** $D_\psi$: the predictive and the chain-rule evidence given θ, trained with the true θ;
- an **NPE proposal** $q_\phi(\theta\mid A,B)$: a conditional flow, trained with the true θ (forward KL, mass-covering);
- **self-normalized IS**, with weights $p(\theta)\hat p(A\mid B,\theta)/q(\theta)$.

The estimator is consistent for any proposal with adequate support. The proposal only changes efficiency (spec §0, §2).

**Setup** (spec §1–§3, d = 1, phase H0 / Z0; deviations listed below).
- **Prior:**
  - $g$ is a random 2-layer tanh MLP (width 32), min–max normalised on 256 points;
  - $T_\theta$ is the time-1 flow of $v_\theta(x) = x(1-x)\sum_j \theta_j\cos((j-1)\pi x)$, with $\theta_j = s_T/j\cdot\tilde\theta_j$,
    $\tilde\theta\sim N(0, I_J)$, $s_T$ calibrated so that $E|T(x)-x|\approx 0.13$, and RK4;
  - $h = \mathrm{id}$;
  - $N_B = 128$, $n_A\in\{1,\dots,10\}$, 32 queries;
  - noise after $h$ (C2′), with $\sigma_A, \sigma_B \sim U[0.005, 0.03]$.
- **Decoder:** 6 layers, $d_{\text{model}} = 128$, 4 heads, a 400-bin bar distribution on $[-0.25, 1.25]$.
  - Masks: B ↔ B; A causal in a random order; x* sees everything; the evidence query $a$ sees B and $A_{<a}$.
  - The null head is the same weights with B masked out.
  - Loss: soft-target CE, trained on a mixture of $J \in \{2,4,8,16\}$ with the true θ.
- **Proposal** (one per J): encoder E0, a 4-seed pooled context $c\in\mathbb R^{256}$, and a zuko NSF (5 transforms,
  hidden 128) over the standardized $\tilde\theta$, trained by NPE. Defensive mixture $\epsilon = 0.1$.
- **Estimator:** K draws; evidence averaged over 2 orders of A.
- **References:**
  - **G-ref**: dense-B grid Bayes, $101^2$ over $\tilde\theta\in[-4,4]^2$, $J = 2$;
  - **IS-ref**: prior IS with the decoder, $K = 4096$, $J = 2$;
  - **IS-exact**: dense-B IS with $q_{\text{mix}}$, $K = 10^4$, $J > 2$;
  - **teacher**: the decoder at the true θ (under H0 this is also the spec's "oracle", since $h = \mathrm{id}$);
  - **A-only**: the null head.
- **Evaluation:** 128 fixed tasks per (J, $n_A$), nested in $n_A \in \{1,2,3,5,8,10\}$, 32 queries each.

**Deviations from the spec (made before the first run).**
- 128 tasks per $n_A$ instead of 256, and IS-ref with $K = 4096$ instead of 8192 (the overnight budget).
- Encoder E1 is not run: it needs the "run-2 transport-token module", which is not in this repo. So the E1-vs-E0
  question of V5 stays open.
- RK4 uses 24 steps instead of 16, so that the $J = 16$ inverse error stays below $10^{-4}$ in float32.
- One decoder serves all J; it is trained on the J-mixture, and it sees θ only through $B_{\text{in}A}$.
- V6–V8 (d = 2, $h$, relatedness) are out of scope tonight.

**What to look for.**
1. V0: the transform family's properties, and G-ref between A-only and the teacher.
2. V1: teacher vs A-only; evidence fidelity on the $41^2$ grid; permutation spread.
3. V2: IS-ref vs G-ref.
4. V3: SBC ranks, HPD coverage, KL(grid ‖ q).
5. V4: q-IS with $K \in \{8, 32, 128\}$; ESS/K; the error decomposition total = (IS-ref − G-ref) + (q-IS − IS-ref).
6. V5: ESS/K of q-IS vs prior IS at $K = 32$ across $J \in \{2,4,8,16\}$, and the NLL gap to IS-exact.

**Pass criteria** (from the spec, §8; nats per query).
- **V0:** T is monotone with fixed endpoints; inverse error $< 10^{-4}$; mean $|T(x)-x| \approx 0.13$; G-ref NLL lies between
  A-only and the teacher/oracle.
- **V1.1:** the teacher improves monotonically with $n_A$, and at $n_A = 10$ is within 0.1 of the oracle (trivial under H0).
- **V1.2:** median evidence KL < 0.1 nats at every $n_A$; report the 95th percentile of over-rating far from the posterior.
- **V1.3:** median permutation spread < 0.05 nats.
- **V2:** IS-ref − G-ref ≤ 0.12 nats at every $n_A$.
- **V3:** SBC KS $p > 0.01$ per coordinate; 50% / 90% HPD coverage within ±5 points; median KL(grid ‖ q) < 0.2 nats.
- **V4:**
  - $\text{NLL}_{q\text{-IS},32} - \text{NLL}_{\text{IS-ref}} \le 0.03$;
  - ESS/K ≥ 0.3;
  - the total gap to G-ref ≤ 0.15 at every $n_A$.
- **V5:**
  - ESS/K of q-IS ≥ 0.2 at $J = 16$ while prior IS collapses (expected < 0.01);
  - the NLL gap to IS-exact grows by ≤ 0.05 from $J = 2$ to $J = 16$.

**Stop rule** (spec §10): if a stage fails, report it with the diagnostics and apply the listed fallback once. Tonight
the stages run in one evaluation pass, so failures are reported per stage, and fallbacks are proposed rather than run.

**Limits.**
- One seed.
- G-ref knows $g$ exactly, while the models see 128 noisy B points: that is worth about 0.1 nats at large $n_A$, so
  G-ref is a floor, not a target.
- d = 1, H0 and Z0 only.


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/air/<sweep_id>"  # set after `just snapshot runs/air/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.